# Model 10: Body fat percentage prediction with multiple linear regression

### Authors
   * Jonathan Sancho Loaiza
   * Jennifer Lopez Miranda
   * Magdaleno Gomez Diaz

## Problem Analysis (CRISP-DM 1: Business Understanding)

The share of a person's weight that is fat is the number a doctor or a trainer actually wants, but measuring it properly is expensive: the reference method weighs the person under water to obtain the density of the body and converts that density to a fat percentage with the Siri equation. The question of this notebook is whether the same percentage can be predicted from measurements anyone can take with a tape and a scale, so that the assistant of the project can answer "what is my body fat" from a few circumferences.

The dataset is the body fat file of Penrose, Nelson and Fisher, distributed on Kaggle, with 252 men. Each row records the underwater density, the fat percentage derived from it, the age, the weight in pounds, the height in inches and ten circumferences in centimetres: neck, chest, abdomen, hip, thigh, knee, ankle, biceps, forearm and wrist.

The technique is multiple linear regression, taught in week four. This dataset is a textbook case of the problem the course calls multicolinealidad, since the circumferences of a body all grow together with its size, so the notebook spends its effort on detecting that problem with the VIF and the sign of the slopes, and on choosing a set of predictors whose slopes can be read.

### Libraries

In [ ]:
# Dataset handling
import os
import kagglehub
import numpy as np
import pandas as pd

# Plots
import matplotlib.pyplot as plt
import seaborn as sns

# Model
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Split, evaluation and persistence
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

%matplotlib inline

RANDOM_STATE = 42

# Folder shared by every model of the project, read later by the REST API
MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

## Data Understanding (CRISP-DM 2)

The dataset is downloaded through kagglehub, which fetches the latest version into a local cache and returns its folder, so the same cell works on a local machine and in Google Colab.

In [ ]:
# Download the dataset and load its only csv file
path = kagglehub.dataset_download('fedesoriano/body-fat-prediction-dataset')
print('Path to dataset files:', path)
bodyfat = pd.read_csv(os.path.join(path, 'bodyfat.csv'))
bodyfat.head()

Each row is one man. `Density` is the underwater measurement, `BodyFat` the percentage derived from it and the target, `Age` in years, `Weight` in pounds, `Height` in inches, and the remaining ten columns are circumferences in centimetres. The first man, 23 years old and 154 pounds, has 12.3 percent fat and an abdomen of 85.2 centimetres.

In [ ]:
# Size, column types and missing values
print("Rows and columns:", bodyfat.shape)
bodyfat.info()
print("Duplicated rows:", bodyfat.duplicated().sum())

There are 252 men and 15 columns, every column is numeric and none has a missing value, and no row is repeated. With 13 candidate predictors and 252 rows the sample is small, which matters twice: the test set will have about 60 men, so the metrics will be noisy, and a model with all 13 predictors will fit noise more easily than one with three.

In [ ]:
# Summary statistics
bodyfat.describe().T.round(2)

The table lists, for each column, the count, the mean, the standard deviation, the minimum, the quartiles and the maximum. The fat percentage runs from 0 to 47.5 with a mean of 19.2 and a median of 19.2, so the target is balanced with no long tail. Weight averages 179 pounds but reaches 363, and the abdomen averages 92.6 centimetres and reaches 148, so there is at least one very large man in the file.

Two minimums are impossible. A height of 29.5 inches is 75 centimetres, the height of a toddler, for a man who weighs 205 pounds, so that value is a typing error, most likely 69.5. A fat percentage of 0.0 is not a measurement either: the Siri equation applied to that man's density of 1.1089 gives a negative number, which was clipped to zero when the file was made. Both rows are shown below and removed, since a regression fitted on a 75 centimetre man learns a height slope from a mistake.

In [ ]:
# The two rows with impossible values
MIN_PLAUSIBLE_HEIGHT = 50
print(bodyfat[bodyfat['Height'] < MIN_PLAUSIBLE_HEIGHT])
print()
print(bodyfat[bodyfat['BodyFat'] <= 0])

bodyfat = bodyfat[(bodyfat['Height'] >= MIN_PLAUSIBLE_HEIGHT) & (bodyfat['BodyFat'] > 0)].reset_index(drop=True)
print()
print("Rows after removing the two data errors:", len(bodyfat))

The two rows are the ones described. The file keeps 250 men. The 363 pound man is kept: his circumferences are all large in proportion, so he is a real observation and not an error, and if the split sends him to the test set the extrapolation check will decide whether the model is allowed to predict him.

## Data Exploration (CRISP-DM 2) and Data Preparation (CRISP-DM 3)

The first decision is what not to use. `Density` is the measurement that `BodyFat` was computed from, through a fixed formula, so a model with `Density` as a predictor would reproduce the target almost exactly and answer nothing, since the whole point is to avoid measuring the density. It is dropped before any model is fitted, and the correlation below shows why.

In [ ]:
# Correlation of every column with the target, Pearson and Spearman side by side
correlation_target = pd.DataFrame({
    'Pearson': bodyfat.corr()['BodyFat'],
    'Spearman': bodyfat.corr(method='spearman')['BodyFat'],
}).drop('BodyFat').sort_values('Pearson', ascending=False).round(3)
print(correlation_target)

Pearson measures how close each relationship is to a straight line and Spearman whether the ranking of one column follows the ranking of the other, both from -1 to 1. `Density` correlates at -0.99 with the fat percentage, which confirms it is the same information and justifies dropping it. Among the real predictors `Abdomen` leads at 0.81 in both Pearson and Spearman, followed by `Chest` at 0.70, `Hip` at 0.61 and `Weight` at 0.60. The two coefficients agree closely for every column, so the relationships are straight enough for a linear model. `Height` is the only column with no relationship at all, at -0.03, because tall men are not fatter than short ones.

In [ ]:
# Drop the leaking column and keep the thirteen tape and scale measurements
CANDIDATES = [c for c in bodyfat.columns if c not in ('BodyFat', 'Density')]
TARGET = 'BodyFat'

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(bodyfat[TARGET], bins=25, ax=axes[0], color='#4c72b0')
axes[0].set_title('Distribution of body fat percentage')
sns.scatterplot(data=bodyfat, x='Abdomen', y=TARGET, ax=axes[1])
axes[1].set_title('Body fat against abdomen circumference')
sns.scatterplot(data=bodyfat, x='Weight', y=TARGET, ax=axes[2])
axes[2].set_title('Body fat against weight')
plt.tight_layout()
plt.show()
print("Imbalance (skew) of BodyFat, 0 means symmetric:", round(bodyfat[TARGET].skew(), 2))

The histogram of the target is a bell centred around 19 percent, and its skew of 0.18 confirms it is symmetric, so unlike a price it needs no transformation. The middle panel shows the abdomen against the fat: the points form a clear upward band, with one man far to the right at 148 centimetres who is the 363 pound man, and even he sits on the extension of the band. The weight panel shows the same upward trend but wider, since a heavy man can be heavy with muscle or with fat, which is why weight correlates less than the abdomen does.

In [ ]:
# Correlation between the thirteen predictors
predictor_correlation = bodyfat[CANDIDATES].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(predictor_correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Pearson correlation between the predictors')
plt.show()

# Every pair above the alarm level of 0.75
ALARM_CORRELATION = 0.75
strong_pairs = [(a, b, round(predictor_correlation.loc[a, b], 3))
                for i, a in enumerate(CANDIDATES) for b in CANDIDATES[i + 1:]
                if abs(predictor_correlation.loc[a, b]) > ALARM_CORRELATION]
print("Pairs of predictors with |r| above", ALARM_CORRELATION, ":", len(strong_pairs))
for a, b, r in sorted(strong_pairs, key=lambda t: -abs(t[2])):
    print("  %-8s %-8s %.3f" % (a, b, r))

The heatmap is almost entirely red. Seventeen pairs of predictors correlate above 0.75, the alarm level used in the course, and `Weight` is in seven of them: it correlates at 0.94 with the hip, 0.89 with the chest and 0.89 with the abdomen. The chest and the abdomen correlate at 0.91 with each other, the hip and the thigh at 0.89. This is anatomy rather than an accident of the sample. A bigger man has a bigger everything, so the thirteen measurements are mostly thirteen readings of one quantity, the size of the body, plus a little specific information each.

For a linear regression this is the problem the course names multicolinealidad. When two predictors carry the same information the model cannot decide how to split the effect between them, the slopes become unstable and can take the wrong sign, and adding or removing one man can move them. The VIF below measures it predictor by predictor.

In [ ]:
# VIF of each predictor against all the others. 1 means independent, 10 is the alarm level
ALARM_VIF = 10
X_candidates = sm.add_constant(bodyfat[CANDIDATES])
vif_table = pd.DataFrame({
    'Predictor': CANDIDATES,
    'VIF': [variance_inflation_factor(X_candidates.values, i + 1) for i in range(len(CANDIDATES))],
}).sort_values('VIF', ascending=False).round(2)
print(vif_table.to_string(index=False))
print()
print("Predictors above the alarm level:", list(vif_table[vif_table['VIF'] > ALARM_VIF]['Predictor']))

The VIF says how much the variance of a slope is inflated because the predictor can be predicted from the others, and a value of 10 means the other predictors explain 90 percent of it. `Weight` scores 44.5, `Hip` 14.4, `Abdomen` 12.5 and `Chest` 10.2, all above the alarm level. A VIF of 44 for weight means that the other twelve measurements predict a man's weight almost perfectly, which is exactly what a set of circumferences should do, and it means a slope for weight in a model with all thirteen predictors cannot be trusted. The correlation matrix flagged the pairs and the VIF confirms that four predictors are redundant with the group as a whole.

In [ ]:
# 75 / 25 split, done once and shared by every model of the notebook
bodyfat_train, bodyfat_test = train_test_split(bodyfat, test_size=0.25, random_state=RANDOM_STATE)
print("Training men:", len(bodyfat_train))
print("Test men:", len(bodyfat_test))

# Remove test men outside the training range of any candidate predictor
inside = pd.Series(True, index=bodyfat_test.index)
for column in CANDIDATES:
    inside &= bodyfat_test[column].between(bodyfat_train[column].min(), bodyfat_train[column].max())
print("Test men removed for falling outside the training range:", int((~inside).sum()))
bodyfat_test = bodyfat_test[inside]
print("Test men kept:", len(bodyfat_test))

The split leaves 187 men for training and 63 for testing, and the extrapolation check removes four test men who have at least one measurement outside the range seen in training. A line has no information beyond the range it was fitted on, and the test set is filtered once here, so both models are scored on the same 59 men.

## Machine Learning Model (CRISP-DM 4: Modeling)

Two models are fitted. Model A uses all thirteen measurements, which is the literal task. Model B is a compact model chosen after watching what multicollinearity does to the slopes of Model A, and both are compared on the same test men.

In [ ]:
# Helper used by every model: fit OLS on the training men and return the result
def fit_ols(features):
    X = sm.add_constant(bodyfat_train[features])
    return sm.OLS(bodyfat_train[TARGET], X).fit()

### Model A: all thirteen measurements

In [ ]:
model_all = fit_ols(CANDIDATES)
print(model_all.summary())

The summary reports an R-squared of 0.774 and an adjusted R-squared of 0.757 on the training men, so the thirteen measurements explain about three quarters of the variation in fat percentage. The middle block is where the problem shows. Only two slopes have a `P>|t|` below 0.05, `Abdomen` at 0.96 points of fat per centimetre and `Wrist` at -2.00 per centimetre. Every other slope, including `Weight`, `Chest`, `Hip` and `Neck`, cannot be told from zero, even though each of them correlates above 0.49 with the target on its own.

The signs are the readable symptom. `Weight`, `Chest`, `Hip`, `Neck` and `Knee` all correlate positively with the fat percentage, and all five have a negative slope in this model. A negative slope for weight says that, holding the twelve other measurements fixed, a heavier man has less fat, which is not nonsense once it is read as "heavier at the same circumferences means denser, so more muscle", but it is not the effect anyone asked for and it is not stable. The next cell shows the flip happening.

In [ ]:
# The sign flip, step by step: the slope of each predictor alone, then with Abdomen beside it
print("%-8s %14s %20s" % ("", "slope alone", "slope with Abdomen"))
for column in ['Weight', 'Chest', 'Hip', 'Neck']:
    alone = fit_ols([column]).params[column]
    paired = fit_ols([column, 'Abdomen']).params[column]
    print("%-8s %14.3f %20.3f" % (column, alone, paired))

Alone, each of the four predictors has a positive slope, weight at 0.195 points per pound and hip at 0.84 per centimetre, matching their positive correlation with the target. The moment `Abdomen` enters the model the four slopes flip negative. The abdomen carries the fat information they were carrying, and what is left for them is the part of their size that is not fat, which pulls the other way. This is the damage that multicollinearity does: the slopes of Model A are not measurements of what each variable is worth, they are the leftovers of a fight over shared information. The two significant `P>|t|` values in the summary did not catch it, since a model can have significant slopes and unreadable ones at once. The VIF and the sign check did.

### Model B: abdomen, weight and wrist

The compact model keeps the abdomen, because it is the strongest single predictor and the site where men store fat, the weight, because the pair abdomen plus weight separates fat from frame, and the wrist, a bone measurement that carries no fat and tells the model how large the skeleton is. Each of the three brings information the other two do not, which is what the VIF should confirm.

In [ ]:
COMPACT = ['Abdomen', 'Weight', 'Wrist']
X_compact = sm.add_constant(bodyfat[COMPACT])
print(pd.DataFrame({
    'Predictor': COMPACT,
    'VIF': [variance_inflation_factor(X_compact.values, i + 1) for i in range(len(COMPACT))],
}).round(2).to_string(index=False))
print()
model_compact = fit_ols(COMPACT)
print(model_compact.summary())

With three predictors the VIF of weight drops from 44.5 to 6.2 and the abdomen from 12.5 to 4.7, both below the alarm level, so the slopes can be read. The training R-squared is 0.760 and the adjusted 0.757, the same adjusted value Model A reached with ten more predictors, which is the adjusted R-squared doing its job: the ten extra measurements were adding noise, not explanation.

All three slopes now have a `P>|t|` below 0.05. Each centimetre of abdomen adds 1.00 points of fat, each pound of weight at the same abdomen removes 0.11, and each centimetre of wrist removes 1.23. The weight slope is still negative, and here that is the intended reading: at a fixed waist, a heavier man carries the extra weight as muscle and bone.

In [ ]:
# Standardized residuals of the compact model, the three usual panels
residuals = model_compact.get_influence().resid_studentized_internal

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(residuals, bins=25, ax=axes[0], color='#4c72b0')
axes[0].set_title('Histogram of standardized residuals')
sns.boxplot(x=residuals, ax=axes[1], color='#4c72b0')
axes[1].set_title('Box plot of standardized residuals')
sm.qqplot(residuals, line='s', ax=axes[2])
axes[2].set_title('Q-Q plot of standardized residuals')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(model_compact.fittedvalues, residuals, alpha=0.6)
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Predicted body fat (%)')
plt.ylabel('Standardized residual')
plt.title('Residuals against predicted value, compact model')
plt.show()
print("Training men with a standardized residual beyond 3:", int((abs(residuals) > 3).sum()))

The standardized residual is each error divided by its own standard deviation, so it is read in standard deviations and a value beyond 3 marks a man the model missed badly. The histogram is a bell centred on zero, the box plot has no dot far from the whiskers, and in the Q-Q plot, which compares the sorted residuals with what a normal distribution would give, the dots follow the red line along its whole length. The residuals against the prediction show a band of constant width from 5 to 35 percent with no fan and no curve, so the spread of the errors does not grow with the prediction and the constant spread assumption holds. No training man is beyond 3 standard deviations. The compact model satisfies every assumption of the technique.

## Evaluation (CRISP-DM 5)

Both models predict the same 59 test men. The metrics are the R-squared, the adjusted R-squared, which corrects for the number of predictors and is the fair comparison between a model with thirteen and a model with three, the mean absolute error in points of fat percentage and the root mean squared error.

In [ ]:
def evaluate(name, model, features):
    y_true = bodyfat_test[TARGET]
    y_pred = model.predict(sm.add_constant(bodyfat_test[features]))
    r2 = r2_score(y_true, y_pred)
    n, p = len(y_true), len(features)
    return pd.Series({
        'R2': r2,
        'Adjusted R2': 1 - (1 - r2) * (n - 1) / (n - p - 1),
        'MAE (% fat)': mean_absolute_error(y_true, y_pred),
        'RMSE (% fat)': np.sqrt(mean_squared_error(y_true, y_pred)),
    }, name=name), y_pred

metrics_all, pred_all = evaluate('Model A: 13 predictors', model_all, CANDIDATES)
metrics_compact, pred_compact = evaluate('Model B: 3 predictors', model_compact, COMPACT)
print(pd.concat([metrics_all, metrics_compact], axis=1).round(3))

On the unseen men Model A reaches an R-squared of 0.602 and Model B 0.583, so the literal model scores 0.02 higher, and its mean absolute error of 3.83 points is 0.04 points lower than Model B's 3.87. That is the cost of deviating, stated before the argument: the thirteen-predictor model is very slightly more accurate on these 59 men. The difference is four hundredths of a point of fat on a test set where one man more or less would move the numbers more than that, and the adjusted R-squared, which charges each predictor for its place, puts the two models at 0.487 and 0.561, in favour of the compact one.

Both R-squared values are well below the training ones, 0.77 and 0.76, which is the usual drop from a small sample and says the honest accuracy of a tape measure model is about 4 points of fat.

In [ ]:
# Real against predicted for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, pred) in zip(axes, [('Model A: 13 predictors', pred_all), ('Model B: 3 predictors', pred_compact)]):
    ax.scatter(bodyfat_test[TARGET], pred, alpha=0.6)
    ax.plot([0, 45], [0, 45], color='red', linewidth=1)
    ax.set_xlabel('Real body fat (%)')
    ax.set_ylabel('Predicted body fat (%)')
    ax.set_title(name)
plt.tight_layout()
plt.show()

Each dot is a test man placed by his real and his predicted fat percentage, and the red diagonal is a perfect prediction. The two clouds are almost the same picture: dots spread evenly around the diagonal from about 8 to 35 percent with no region where one model is systematically off. Nothing in the plot separates the models, which is what the metrics said.

## Results (CRISP-DM 6: Deployment)

The model kept is Model B, a multiple linear regression with the abdomen circumference, the weight and the wrist circumference as predictors. On the 59 test men inside the training range it reaches an R-squared of 0.583, an adjusted R-squared of 0.561, a mean absolute error of 3.87 points of fat and a root mean squared error of 4.64. The literal model with all thirteen measurements was built and scored first and reaches 0.602, 0.487, 3.83 and 4.53 on the same men.

The trade is explicit. Model B gives up 0.04 points of average error, on a test set too small to tell that difference from chance, in exchange for three slopes that can be read and defended, a VIF below the alarm level for every predictor, and a measurement protocol of one tape reading of the waist, one of the wrist and a scale, instead of ten circumferences. Model A's slopes are unreadable: five of them contradict the sign of their own correlation with the target, the VIF of weight is 44, and the flip was shown happening as soon as the abdomen entered the model. Significance did not catch any of that, since Model A still has two significant slopes. The VIF and the sign check did.

The limitation to state is the accuracy itself. An average error near 4 points of fat means a man predicted at 20 percent may well be at 16 or 24, which is enough to place someone in a broad category but not to track a diet week by week. The dataset is also 250 men only, so the model says nothing about women. The model is saved below with its three predictors so the REST API can load it without training.

In [ ]:
bundle = {
    'model': model_compact,
    'features': COMPACT,
    'target': TARGET,
    'units': {'Abdomen': 'cm', 'Weight': 'lb', 'Wrist': 'cm'},
    'training_range': {c: (float(bodyfat_train[c].min()), float(bodyfat_train[c].max())) for c in COMPACT},
}
MODEL_PATH = os.path.join(MODELS_DIR, 'body_fat_regression.joblib')
joblib.dump(bundle, MODEL_PATH)
print("Saved to", MODEL_PATH)

In [ ]:
# Reload and predict one man by hand: abdomen 90 cm, weight 175 lb, wrist 18 cm
loaded = joblib.load(MODEL_PATH)
example = pd.DataFrame([{'Abdomen': 90.0, 'Weight': 175.0, 'Wrist': 18.0}])
predicted = loaded['model'].predict(sm.add_constant(example, has_constant='add'))[0]
print("Predicted body fat: %.1f %%" % predicted)

The reloaded model gives 17.3 percent for a man of average build, close to the dataset median of 19.2, which is where an average man should land. This is the call the REST API will make: three numbers in, one percentage out, with the average error of about 4 points reported beside it.